# pipe_catedra/02 — Feature Engineering (porteo de `fe_2.ipynb` / z302)

Input  : `z301_preprocessed_{modo}.parquet`
Output : `z302_features_{modo}.parquet` + `z302_inferencia_{modo}.parquet`

Porteo fiel del notebook de la catedra -- misma logica, mismos nombres de
columnas y de PARAM. Todo el feature engineering ya esta vectorizado con
operaciones de ventana de polars (`.shift()`/`.rolling_mean()`/`.over()`),
no hay loops de Python que valga la pena reescribir con DuckDB aca.

**Cambio real vs el original**: se elimina la ultima celda (seccion 11,
"Verificacion"), que referenciaba una columna `target` que no existe en
este dataset (las columnas reales son `target_nivel`/`target_delta`) -- no
corria.

Responsabilidades:
- Lags consecutivos de `tn` (lag_0 = mes actual, lag_1 ... lag_N)
- Features de tendencia, comportamiento/intermitencia y calendario
- Escalado rolling (media, std) calculado hacia atras (sin leakage)
- Encoding de variables categoricas (cat1, cat2, cat3, brand) para LGBM
- Target = tn en t+horizonte (nivel y delta, las dos columnas)
- Split train / inferencia


## 0) Setup


In [ ]:
import os
from pathlib import Path

import polars as pl
import numpy as np


def resolver_bucket() -> Path:
    """VM de la catedra -> ~/buckets/b1 | Colab -> /content/buckets/b1 | local -> env."""
    env = os.environ.get("LABO3_BUCKET")
    if env and Path(env).expanduser().exists():
        return Path(env).expanduser().resolve()
    for cand in (Path.home() / "buckets" / "b1",
                 "/content/buckets/b1",
                 "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    raise RuntimeError(
        "No encontre el bucket. Defini LABO3_BUCKET, ej: "
        "os.environ['LABO3_BUCKET'] = '/home/usuario/labo3-bucket'"
    )


BUCKET  = resolver_bucket()
DIR_OUT = BUCKET / "exp_pipe_catedra"
DIR_OUT.mkdir(parents=True, exist_ok=True)
print(f"BUCKET: {BUCKET}")
print(f"salida: {DIR_OUT}")


## 1) Parametros — palancas


In [ ]:
PARAM = {
    'experimento': 'z302',

    # PALANCA 1 (debe coincidir con 01_)
    'modo_agrupacion': 'producto',

    # PALANCA 3: lags consecutivos
    'lags': list(range(0, 12)),

    # PALANCA 4: ventana rolling
    'ventana_rolling': None,

    # PALANCA 5: tipo de target
    'tipo_target': 'nivel',

    # PALANCA 6: market share
    'incluir_market_share': True,

    # PALANCA 7: horizonte
    'horizonte': 2,

    'cols_categoricas': ['cat1', 'cat2', 'cat3', 'brand'],
}

MODO = PARAM['modo_agrupacion']
PARAM['path_input']  = str(DIR_OUT / f"z301_preprocessed_{MODO}.parquet")
PARAM['path_output'] = str(DIR_OUT / f"z302_features_{MODO}.parquet")

print('Parametros:', PARAM)
print('Input:',  PARAM['path_input'])
print('Output:', PARAM['path_output'])


## 2) Carga


In [ ]:
df = pl.read_parquet(PARAM['path_input'])
print(f'Input: {df.shape}')
print(df.schema)
df.head(3)


## 3) Market share y categoria (vienen de 01_)

Estas features se calculan en `01_Preprocesamiento` sobre el universo completo. Aca solo se verifican.


In [ ]:
for c in ['tn_total_cat2', 'market_share', 'productos_activos_cat2', 'productos_nuevos_cat2_3m']:
    estado = 'OK' if c in df.columns else 'FALTA (re-correr 01_)'
    print(f'  {c}: {estado}')


## 4) Encoding de categoricas para LGBM

LGBM no acepta strings. Convertimos cat1/cat2/cat3/brand a codigos enteros con `.to_physical()`.
El mapeo es consistente porque se aplica sobre todo el dataset de una vez.


In [ ]:
for col in PARAM['cols_categoricas']:
    if col in df.columns:
        df = df.with_columns(
            pl.col(col).cast(pl.Categorical).to_physical().cast(pl.Int32).alias(col)
        )
        print(f'  {col}: encodeado')

if 'descripcion' in df.columns:
    df = df.drop('descripcion')
    print('  descripcion: eliminada (texto libre)')

print('Encoding completo.')


## 5) Lags consecutivos de tn

`lag_0` = mes actual (= tn, sin desplazar). `lag_k` = tn de hace k meses.
Calculados dentro de cada `agrupa_id`.


In [ ]:
df = df.sort(['agrupa_id', 'periodo'])

lag_exprs = [
    pl.col('tn').shift(k).over('agrupa_id').alias(f'lag_{k}')
    for k in PARAM['lags']
]
df = df.with_columns(lag_exprs)

print('Lags calculados:', [f'lag_{k}' for k in PARAM['lags']])
df.filter(pl.col('agrupa_id') == df['agrupa_id'][0]).select(
    ['periodo', 'tn'] + [f'lag_{k}' for k in PARAM['lags'][:4]]
).head(6)


## 5b) Features de tendencia

Diferencias, pendientes, ratios y medias moviles. Le dan al modelo señal de **direccion** (si el producto sube o baja), atacando el sesgo de sobreestimacion en productos en declive.

- `delta_1/2/3`: cambio respecto a 1, 2, 3 meses atras
- `delta_anual`: cambio respecto a ~1 año (lag_11)
- `pendiente_3m`: tendencia de los ultimos 3 meses
- `ratio_0_1`, `ratio_0_3`: crecimiento relativo
- `media_movil_3/6`: medias de 3 y 6 meses (sin leakage)
- `ms_delta_3`: cambio del market share vs 3 meses atras


In [ ]:
df = df.sort(['agrupa_id', 'periodo'])

df = df.with_columns([
    (pl.col('lag_0') - pl.col('lag_1')).alias('delta_1'),
    (pl.col('lag_0') - pl.col('lag_2')).alias('delta_2'),
    (pl.col('lag_0') - pl.col('lag_3')).alias('delta_3'),
    (pl.col('lag_0') - pl.col('lag_11')).alias('delta_anual'),
])

df = df.with_columns(
    ((pl.col('lag_0') - pl.col('lag_2')) / 2).alias('pendiente_3m')
)

df = df.with_columns([
    pl.when(pl.col('lag_1') > 0).then(pl.col('lag_0') / pl.col('lag_1'))
      .otherwise(pl.lit(0.0)).alias('ratio_0_1'),
    pl.when(pl.col('lag_3') > 0).then(pl.col('lag_0') / pl.col('lag_3'))
      .otherwise(pl.lit(0.0)).alias('ratio_0_3'),
])

df = df.with_columns([
    pl.col('tn').shift(1).rolling_mean(window_size=3, min_periods=1)
      .over('agrupa_id').alias('media_movil_3'),
    pl.col('tn').shift(1).rolling_mean(window_size=6, min_periods=1)
      .over('agrupa_id').alias('media_movil_6'),
])

if 'market_share' in df.columns:
    df = df.with_columns(
        (pl.col('market_share') - pl.col('market_share').shift(3).over('agrupa_id'))
        .alias('ms_delta_3')
    )

print('Features de tendencia calculadas.')
cols_nuevas = ['delta_1', 'delta_3', 'delta_anual', 'pendiente_3m',
               'ratio_0_1', 'media_movil_3', 'media_movil_6']
_prod_muestra = df['product_id'][0] if 'product_id' in df.columns else None
print(df.filter(pl.col('product_id') == _prod_muestra).select(['periodo', 'tn'] + cols_nuevas).head(6))


## 5c) Features de comportamiento e intermitencia

Capturan patrones de actividad y estabilidad de la serie (calculadas sobre `agrupa_id`, funcionan en ambos modos):
- `meses_activo`: meses desde la primera venta real
- `racha_ceros`: meses consecutivos en cero hasta el actual (intermitencia)
- `compro_igual_mes_ant`: si vendio exactamente lo mismo que el mes anterior
- `racha_crece` / `racha_cae`: meses consecutivos subiendo / bajando
- `pct_ceros_hist`: proporcion de meses en cero en toda la historia

La racha de ceros ataca productos en declive/intermitentes, donde el modelo sobreestima.


In [ ]:
df = df.sort(['agrupa_id', 'periodo'])

# 1) Meses activo: cantidad de meses transcurridos desde la primera venta real (tn>0)
df = df.with_columns(
    pl.int_range(1, pl.len() + 1).over('agrupa_id').cast(pl.Int32).alias('meses_activo')
)

# 2) Vendio exactamente lo mismo que el mes anterior?
df = df.with_columns(
    (pl.col('tn') == pl.col('tn').shift(1).over('agrupa_id'))
    .cast(pl.Int8).fill_null(0).alias('compro_igual_mes_ant')
)

# 3) Racha de ceros: meses consecutivos en cero hasta el actual
es_cero = (pl.col('tn') == 0).cast(pl.Int32)
df = df.with_columns(es_cero.alias('_es_cero'))
df = df.with_columns(
    (1 - pl.col('_es_cero')).cum_sum().over('agrupa_id').alias('_bloque')
)
df = df.with_columns(
    pl.col('_es_cero').cum_sum().over(['agrupa_id', '_bloque']).cast(pl.Int32).alias('racha_ceros')
)

# 4) Rachas de tendencia: meses consecutivos subiendo o bajando
sube = (pl.col('tn') > pl.col('tn').shift(1)).over('agrupa_id')
baja = (pl.col('tn') < pl.col('tn').shift(1)).over('agrupa_id')
df = df.with_columns([
    sube.cast(pl.Int8).fill_null(0).alias('_sube'),
    baja.cast(pl.Int8).fill_null(0).alias('_baja'),
])
df = df.with_columns([
    (1 - pl.col('_sube')).cum_sum().over('agrupa_id').alias('_blq_sube'),
    (1 - pl.col('_baja')).cum_sum().over('agrupa_id').alias('_blq_baja'),
])
df = df.with_columns([
    pl.col('_sube').cum_sum().over(['agrupa_id', '_blq_sube']).cast(pl.Int32).alias('racha_crece'),
    pl.col('_baja').cum_sum().over(['agrupa_id', '_blq_baja']).cast(pl.Int32).alias('racha_cae'),
])

# 5) Proporcion historica de ceros hasta el periodo actual (intermitencia acumulada)
df = df.with_columns(
    (pl.col('_es_cero').cum_sum().over('agrupa_id') /
     pl.int_range(1, pl.len() + 1).over('agrupa_id'))
    .cast(pl.Float32).alias('pct_ceros_hist')
)

df = df.drop(['_es_cero', '_bloque', '_sube', '_baja', '_blq_sube', '_blq_baja'])

print('Features de comportamiento calculadas.')
cols_comp = ['meses_activo', 'racha_ceros', 'compro_igual_mes_ant',
             'racha_crece', 'racha_cae', 'pct_ceros_hist']
print(df.filter(pl.col('product_id') == _prod_muestra).select(['periodo', 'tn'] + cols_comp).head(8))


## 6) Escalado: media y desvio rolling

Calculado hacia atras (con `shift(1)` para no incluir el periodo actual -> sin leakage).
`media_rolling` y `std_rolling` quedan como features.
`tn_scaled = tn / media_rolling` (feature adicional, no reemplaza tn).


In [ ]:
ventana = PARAM['ventana_rolling']
w = len(df) if ventana is None else ventana

df = df.with_columns([
    pl.col('tn').shift(1).rolling_mean(window_size=w, min_periods=1)
      .over('agrupa_id').alias('media_rolling'),
    pl.col('tn').shift(1).rolling_std(window_size=w, min_periods=2)
      .over('agrupa_id').alias('std_rolling'),
])

df = df.with_columns(
    pl.when(pl.col('media_rolling') > 0)
      .then(pl.col('tn') / pl.col('media_rolling'))
      .otherwise(pl.lit(0.0))
      .alias('tn_scaled')
)

print('Escalado calculado.')
df.filter(pl.col('product_id') == _prod_muestra).select(
    ['periodo', 'tn', 'media_rolling', 'std_rolling', 'tn_scaled']
).head(8)


## 7) Features de calendario


In [ ]:
df = df.with_columns([
    (pl.col('periodo') % 100).cast(pl.Int8).alias('mes'),
    (pl.col('periodo') // 100).cast(pl.Int16).alias('anio'),
])
print('Features de calendario: mes, anio')


## 8) Targets: nivel y delta (ambos como columnas)

Se calculan **los dos** targets siempre. `03_Optuna` elige cual usar con la palanca `tipo_target`.
- `target_nivel` = tn en t+2
- `target_delta` = tn(t+2) - tn(t)


In [ ]:
h = PARAM['horizonte']

df = df.with_columns(
    pl.col('tn').shift(-h).over('agrupa_id').alias('tn_t2')
)

df = df.with_columns([
    pl.col('tn_t2').alias('target_nivel'),
    (pl.col('tn_t2') - pl.col('tn')).alias('target_delta'),
])

df_train = df.filter(pl.col('tn_t2').is_not_null())
df_infer = df.filter(pl.col('tn_t2').is_null())

print(f'Train: {df_train.height:,} filas')
print(f'Inferencia: {df_infer.height:,} filas')
print('Targets disponibles: target_nivel, target_delta')


## 9) Limpieza

`lag_0` nunca es null (= tn). No removemos filas por lags largos nulos: LGBM los maneja nativamente y removerlas perderia productos con historia corta.


In [ ]:
for k in [0, min([l for l in PARAM['lags'] if l > 0])]:
    nulos = df_train.filter(pl.col(f'lag_{k}').is_null()).height
    print(f'  lag_{k}: {nulos} nulls en train')
print(f'Train final: {df_train.height:,} filas')


## 10) Guardar outputs


In [ ]:
df_train.write_parquet(PARAM['path_output'])
print(f'Train guardado: {PARAM["path_output"]}')
print(f'   Filas: {df_train.height:,}  |  Columnas: {len(df_train.columns)}')
print('   Columnas:', df_train.columns)

path_infer = PARAM['path_output'].replace('z302_features', 'z302_inferencia')
df_infer.write_parquet(path_infer)
print(f'Inferencia guardada: {path_infer}')
print(f'   Filas: {df_infer.height:,}')
